# 03. Pemodelan Support Vector Regression

SVR memakai input lag-1 (hasil PACF skripsi) dan grid search pada C = 2^0..2^9, gamma = 2^-3..2^3, epsilon = 0..1 (Tabel 4.7 skripsi). Berbeda dari skripsi, pemilihan hyperparameter memakai **TimeSeriesSplit** sehingga data testing benar-benar tidak pernah dilihat saat tuning.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from kurs_forecast.data import load_kurs, split_train_test
serie = load_kurs(); train, test = split_train_test(serie)
print(f"Total {len(serie)} minggu: training {len(train)}, testing {len(test)}")

Total 105 minggu: training 84, testing 21


In [2]:
from kurs_forecast.models import fit_svr, predict_svr
from kurs_forecast.metrics import all_metrics
hasil = {}
for kernel in ['rbf', 'poly']:
    r = fit_svr(train, kernel=kernel)
    pred = predict_svr(r, train, test)
    hasil[kernel] = (r, pred)
    print(kernel, r.best_params, 'CV RMSE', round(r.cv_rmse, 2))
    print('   train', {k: round(v, 3) for k, v in r.metrik_train.items()})
    print('   test ', {k: round(v, 3) for k, v in all_metrics(test.values, pred).items()})

rbf {'C': 512.0, 'epsilon': 0.3, 'gamma': 0.125} CV RMSE 176.2
   train {'RMSE': 118.489, 'MAE': 91.667, 'MAPE': 0.616}
   test  {'RMSE': 93.6, 'MAE': 76.871, 'MAPE': 0.509}


poly {'C': 2.0, 'epsilon': 0.5, 'gamma': 0.125} CV RMSE 312.51
   train {'RMSE': 436.268, 'MAE': 385.316, 'MAPE': 2.605}
   test  {'RMSE': 413.373, 'MAE': 390.932, 'MAPE': 2.57}


Seperti skripsi, kernel RBF jauh lebih baik daripada polynomial. Nilai parameter terbaik berbeda dari skripsi (C = 2^4, gamma = 1, epsilon = 0,1) karena target distandardisasi dan validasi silangnya berbeda, tetapi akurasi testing-nya setara.

In [3]:
from kurs_forecast.plots import plot_fit_train
plot_fit_train(train, hasil['rbf'][0].fitted_train, 'SVR RBF', offset=1);

In [4]:
plt.figure(figsize=(10,4))
plt.plot(test.index, test.values, 'k-o', ms=4, label='Aktual')
plt.plot(test.index, hasil['rbf'][1], '--^', color='#f59e0b', ms=3, label='SVR RBF')
plt.plot(test.index, hasil['poly'][1], '--', color='#16a34a', label='SVR polynomial')
plt.legend(); plt.grid(alpha=.3); plt.title('Prediksi testing SVR');

## Catatan penting: satu langkah vs rekursif
Prediksi di atas memakai nilai aktual minggu sebelumnya sebagai input (one-step-ahead), persis seperti skripsi. Jika model harus meramal 21 minggu tanpa melihat aktual (rekursif), galatnya menumpuk. Ini penting dijelaskan ke pemangku kepentingan agar akurasi 0,5% tidak disalahartikan sebagai akurasi peramalan jangka panjang.

In [5]:
pred_rek = predict_svr(hasil['rbf'][0], train, test, recursive=True)
print('One-step :', {k: round(v, 3) for k, v in all_metrics(test.values, hasil['rbf'][1]).items()})
print('Rekursif :', {k: round(v, 3) for k, v in all_metrics(test.values, pred_rek).items()})

One-step : {'RMSE': 93.6, 'MAE': 76.871, 'MAPE': 0.509}
Rekursif : {'RMSE': 554.631, 'MAE': 501.889, 'MAPE': 3.294}
